# Decision Tree and k-Nearest Neighbors Classification

This notebook trains and evaluates Decision Tree and k-Nearest Neighbors (k-NN) classifiers for the Spot the Bot project. Both models use the team's frozen HC3 train, validation, and test splits and TF-IDF text features.

Model settings are selected using the validation set. The test set is used only for final evaluation. Macro-F1 is treated as the primary evaluation metric because the dataset contains more human-written than AI-generated examples.

## 1. Setup and Imports

Import the libraries and shared project utilities needed for feature extraction, model training, and evaluation.

In [1]:
import sys
print(sys.executable)
print(sys.version)

/Users/Melinda/Desktop/Chewy-1A-spot-the-bot-detecting-ai-generated-text/.venv/bin/python
3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))

from spotbot.prepared import load_prepared_splits
from spotbot.evaluation import evaluate_predictions

## 2. Load the Prepared Dataset Splits

Load the team's frozen train, validation, and test splits. These splits were created during the September data-preparation milestone and include leakage checks, so no new split is created in this notebook.

In [3]:
train_df, val_df, test_df = load_prepared_splits()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

Train: (65401, 8)
Validation: (8218, 8)
Test: (8151, 8)


,split,question_group_id,component_id,question,source,text,label,answer_index
0,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,A lot of it has to do with rising third world ...,0,0
1,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,URL_0 Here 's an answer to it being relative t...,0,1
2,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,Because we continued to pay what they are aski...,0,2
3,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,There are several reasons why gas prices have ...,1,0
4,train,qg_000299619139c47d,ac_ee17192f440cf24a,How accurate is the statement that we are all ...,reddit_eli5,Not every element . The hydrogen on Earth coul...,0,0


The prepared dataset contains:

- **65,401 training examples**
- **8,218 validation examples**
- **8,151 test examples**

Using the same frozen splits across models ensures that the Decision Tree, k-NN, and other team models can be compared fairly.

## 3. TF-IDF Feature Extraction

Convert the text into numerical TF-IDF features. The vectorizer is fit only on the training text to avoid information leakage. The learned training vocabulary is then applied to the validation and test sets.

In [4]:
tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Learn the vocabulary ONLY from training data
X_train = tfidf.fit_transform(train_df["text"])

# Apply the same vocabulary to validation and test
X_val = tfidf.transform(val_df["text"])
X_test = tfidf.transform(test_df["text"])

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nTrain labels:")
print(pd.Series(y_train).value_counts())

X_train: (65401, 50000)
X_val: (8218, 50000)
X_test: (8151, 50000)

Train labels:
0    44473
1    20928
Name: count, dtype: int64


The TF-IDF representation contains **50,000 features** for each example.

The training set contains **44,473 human-written examples** and **20,928 AI-generated examples**, confirming that the dataset is imbalanced toward human text. Because of this imbalance, macro-F1 is more informative than accuracy alone.

## 4. Decision Tree Model

### 4.1 Tune Maximum Tree Depth

The first Decision Tree experiment compares several values of `max_depth`. All other settings are held constant so that the effect of tree depth can be evaluated using validation macro-F1.

In [5]:
dt_results = []

depths = [10, 20, 30, 50, None]

for depth in depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    scores = evaluate_predictions(
        y_val,
        val_pred,
        val_df["source"]
    )

    dt_results.append({
        "max_depth": str(depth),
        "macro_f1": scores["macro_f1"],
        "accuracy": scores["accuracy"]
    })

dt_results_df = pd.DataFrame(dt_results)

display(
    dt_results_df.sort_values(
        "macro_f1",
        ascending=False
    )
)

,max_depth,macro_f1,accuracy
1,20,0.893560,0.906425
2,30,0.890964,0.903505
3,50,0.881955,0.894622
4,None,0.880303,0.892796
0,10,0.865171,0.882088


### Depth Tuning Results

A maximum depth of **20** produced the strongest validation performance:

- **Macro-F1: 0.8936**
- **Accuracy: 0.9064**

Shallower trees underfit the data, while increasing the depth beyond 20 slightly reduced validation performance, suggesting that deeper trees begin to overfit.

### 4.2 Tune Minimum Samples per Leaf

With `max_depth=20` fixed, the next experiment varies `min_samples_leaf` to control how many training examples must be present in a terminal leaf.

In [6]:
best_depth = 20

leaf_results = []

for leaf in [1, 2, 5, 10, 20]:
    model = DecisionTreeClassifier(
        max_depth=best_depth,
        min_samples_leaf=leaf,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    scores = evaluate_predictions(
        y_val,
        val_pred,
        val_df["source"]
    )

    leaf_results.append({
        "min_samples_leaf": leaf,
        "macro_f1": scores["macro_f1"],
        "accuracy": scores["accuracy"]
    })

leaf_results_df = pd.DataFrame(leaf_results)

display(
    leaf_results_df.sort_values(
        "macro_f1",
        ascending=False
    )
)

,min_samples_leaf,macro_f1,accuracy
3,10,0.894769,0.907398
2,5,0.893560,0.906425
0,1,0.893445,0.906668
1,2,0.893052,0.906060
4,20,0.885695,0.899002


### Leaf Size Tuning Results

The strongest validation result was obtained with:

- `max_depth = 20`
- `min_samples_leaf = 10`

This configuration achieved:

- **Macro-F1: 0.8948**
- **Accuracy: 0.9074**

A moderate leaf size slightly improved generalization compared with smaller leaves, while a leaf size of 20 reduced performance.

### 4.3 Final Decision Tree Validation Performance

Train the Decision Tree using the selected validation settings and examine its class-level, confusion-matrix, and domain-level performance.

In [7]:
best_dt = DecisionTreeClassifier(
    max_depth=20,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42
)

best_dt.fit(X_train, y_train)

dt_val_pred = best_dt.predict(X_val)

dt_val_scores = evaluate_predictions(
    y_val,
    dt_val_pred,
    val_df["source"]
)

print("FINAL DECISION TREE - VALIDATION")
print("Macro-F1:", dt_val_scores["macro_f1"])
print("Accuracy:", dt_val_scores["accuracy"])

display(dt_val_scores["per_class"])

print("Confusion Matrix:")
print(dt_val_scores["confusion_matrix"])

print("\nPer-domain:")
dt_val_scores["per_domain"]

FINAL DECISION TREE - VALIDATION
Macro-F1: 0.8947688726498342
Accuracy: 0.9073983937697737


,precision,recall,f1,support
class,,,,
human,0.940489,0.922141,0.931225,5587
ai,0.841241,0.876093,0.858313,2631


Confusion Matrix:
[[5152  435]
 [ 326 2305]]

Per-domain:


{'finance': {'macro_f1': 0.9055303419474717,
  'accuracy': 0.906508875739645,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.928962  0.865140  0.895916      393
  ai      0.889353  0.942478  0.915145      452,
  'confusion_matrix': array([[340,  53],
         [ 26, 426]])},
 'medicine': {'macro_f1': 0.9560749001702277,
  'accuracy': 0.9561752988047809,
  'per_class':        precision  recall        f1  support
  class                                      
  human   1.000000   0.912  0.953975      125
  ai      0.919708   1.000  0.958175      126,
  'confusion_matrix': array([[114,  11],
         [  0, 126]])},
 'open_qa': {'macro_f1': 0.8185281904191104,
  'accuracy': 0.8471337579617835,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.642424  0.890756  0.746479      119
  ai      0.957516  0.832386  0.890578      352,
  'confusion_matrix': a

### Decision Tree Validation Analysis

The final Decision Tree achieved:

- **Macro-F1: 0.8948**
- **Accuracy: 0.9074**
- **Human F1: 0.9312**
- **AI F1: 0.8583**

The model performs better on human-written text than AI-generated text, but still identifies most AI examples correctly.

The confusion matrix shows:

- **5,152** human examples correctly classified as human
- **435** human examples incorrectly classified as AI
- **326** AI examples incorrectly classified as human
- **2,305** AI examples correctly classified as AI

Performance also varies substantially by domain. The strongest validation performance occurs on **medicine** with a macro-F1 of approximately **0.956**, while **wiki_csai** is the most difficult domain at approximately **0.762**.

## 5. k-Nearest Neighbors Model

### 5.1 Tune the Number of Neighbors

The k-NN classifier uses cosine distance on the TF-IDF vectors. Several values of `k` are compared using validation macro-F1.

In [8]:
knn_results = []

for k in [5, 11, 21]:

    model = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance",
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    scores = evaluate_predictions(
        y_val,
        val_pred,
        val_df["source"]
    )

    knn_results.append({
        "k": k,
        "macro_f1": scores["macro_f1"],
        "accuracy": scores["accuracy"]
    })

knn_results_df = pd.DataFrame(knn_results)

display(
    knn_results_df.sort_values(
        "macro_f1",
        ascending=False
    )
)

,k,macro_f1,accuracy
2,21,0.869066,0.879654
1,11,0.836832,0.847165
0,5,0.812677,0.821976


### Number of Neighbors Results

Of the tested values, **k = 21** performed best:

- **Macro-F1: 0.8691**
- **Accuracy: 0.8797**

Performance improved as the number of neighbors increased from 5 to 21, suggesting that using a larger neighborhood produced more stable classifications.

### 5.2 Compare Neighbor Weighting

Using `k=21`, compare uniform voting with distance-weighted voting. Distance weighting gives closer neighbors greater influence over the final prediction.

In [9]:
BEST_K = 21

weight_results = []

for weight_type in ["uniform", "distance"]:

    model = KNeighborsClassifier(
        n_neighbors=BEST_K,
        weights=weight_type,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    scores = evaluate_predictions(
        y_val,
        val_pred,
        val_df["source"]
    )

    weight_results.append({
        "weights": weight_type,
        "macro_f1": scores["macro_f1"],
        "accuracy": scores["accuracy"]
    })

weight_results_df = pd.DataFrame(weight_results)

display(
    weight_results_df.sort_values(
        "macro_f1",
        ascending=False
    )
)

,weights,macro_f1,accuracy
1,distance,0.869066,0.879654
0,uniform,0.867947,0.878681


### Weighting Results

Distance weighting slightly outperformed uniform weighting:

- **Distance-weighted macro-F1: 0.8691**
- **Uniform macro-F1: 0.8679**

The final k-NN model therefore uses `k=21`, cosine distance, and distance-weighted voting.

### 5.3 Final k-NN Validation Performance

Evaluate the selected k-NN configuration using the same shared evaluation metrics as the Decision Tree.

In [10]:
best_knn = KNeighborsClassifier(
    n_neighbors=21,
    weights="distance",
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

best_knn.fit(X_train, y_train)

knn_val_pred = best_knn.predict(X_val)

knn_val_scores = evaluate_predictions(
    y_val,
    knn_val_pred,
    val_df["source"]
)

print("FINAL k-NN - VALIDATION")
print("Macro-F1:", knn_val_scores["macro_f1"])
print("Accuracy:", knn_val_scores["accuracy"])

display(knn_val_scores["per_class"])

print("Confusion Matrix:")
print(knn_val_scores["confusion_matrix"])

print("\nPer-domain:")
knn_val_scores["per_domain"]

FINAL k-NN - VALIDATION
Macro-F1: 0.8690658264001383
Accuracy: 0.8796544171331224


,precision,recall,f1,support
class,,,,
human,0.962762,0.856095,0.906300,5587
ai,0.752615,0.929685,0.831831,2631


Confusion Matrix:
[[4783  804]
 [ 185 2446]]

Per-domain:


{'finance': {'macro_f1': 0.7756985388361959,
  'accuracy': 0.7928994082840237,
  'per_class':        precision    recall        f1  support
  class                                        
  human   1.000000  0.554707  0.713584      393
  ai      0.720893  1.000000  0.837813      452,
  'confusion_matrix': array([[218, 175],
         [  0, 452]])},
 'medicine': {'macro_f1': 0.9760770110560426,
  'accuracy': 0.9760956175298805,
  'per_class':        precision  recall        f1  support
  class                                      
  human   1.000000   0.952  0.975410      125
  ai      0.954545   1.000  0.976744      126,
  'confusion_matrix': array([[119,   6],
         [  0, 126]])},
 'open_qa': {'macro_f1': 0.8006972470111218,
  'accuracy': 0.8577494692144374,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.760000  0.638655  0.694064      119
  ai      0.884097  0.931818  0.907331      352,
  'confusion_matrix': 

### k-NN Validation Analysis

The final k-NN model achieved:

- **Macro-F1: 0.8691**
- **Accuracy: 0.8797**
- **Human F1: 0.9063**
- **AI F1: 0.8318**

k-NN achieved especially high **AI recall of 0.9297**, meaning it successfully identified most AI-generated examples. However, this came at the cost of lower human recall.

Its confusion matrix shows:

- **4,783** human examples correctly classified as human
- **804** human examples incorrectly classified as AI
- **185** AI examples incorrectly classified as human
- **2,446** AI examples correctly classified as AI

Like the Decision Tree, k-NN performs especially well on **medicine** but struggles more on **wiki_csai** and **finance**.

## 6. Final Test Evaluation

After selecting all hyperparameters using the validation set, evaluate both final models on the held-out test set. No model settings are changed after examining these results.

In [11]:
dt_test_pred = best_dt.predict(X_test)

dt_test_scores = evaluate_predictions(
    y_test,
    dt_test_pred,
    test_df["source"]
)

print("DECISION TREE - FINAL TEST")
print("Macro-F1:", dt_test_scores["macro_f1"])
print("Accuracy:", dt_test_scores["accuracy"])

display(dt_test_scores["per_class"])

print("Confusion Matrix:")
print(dt_test_scores["confusion_matrix"])

print("\nPer-domain:")
dt_test_scores["per_domain"]

DECISION TREE - FINAL TEST
Macro-F1: 0.8917610321278955
Accuracy: 0.904919641761747


,precision,recall,f1,support
class,,,,
human,0.936229,0.922868,0.929501,5536
ai,0.841500,0.866922,0.854021,2615


Confusion Matrix:
[[5109  427]
 [ 348 2267]]

Per-domain:


{'finance': {'macro_f1': 0.9079110705048995,
  'accuracy': 0.9086595492289442,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.924933  0.875635  0.899609      394
  ai      0.895745  0.937639  0.916213      449,
  'confusion_matrix': array([[345,  49],
         [ 28, 421]])},
 'medicine': {'macro_f1': 0.9413093920138309,
  'accuracy': 0.9418604651162791,
  'per_class':        precision    recall        f1  support
  class                                        
  human   1.000000  0.879032  0.935622      124
  ai      0.899329  1.000000  0.946996      134,
  'confusion_matrix': array([[109,  15],
         [  0, 134]])},
 'open_qa': {'macro_f1': 0.8290768708637366,
  'accuracy': 0.8565400843881856,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.654545  0.907563  0.760563      119
  ai      0.964401  0.839437  0.897590      355,
  'confusion_m

### Decision Tree Test Results

The Decision Tree achieved:

- **Macro-F1: 0.8918**
- **Accuracy: 0.9049**
- **Human Precision: 0.9362**
- **Human Recall: 0.9229**
- **AI Precision: 0.8415**
- **AI Recall: 0.8669**

The test performance is very close to the validation performance, suggesting that the selected Decision Tree settings generalize well to unseen data.

In [12]:
knn_test_pred = best_knn.predict(X_test)

knn_test_scores = evaluate_predictions(
    y_test,
    knn_test_pred,
    test_df["source"]
)

print("k-NN - FINAL TEST")
print("Macro-F1:", knn_test_scores["macro_f1"])
print("Accuracy:", knn_test_scores["accuracy"])

display(knn_test_scores["per_class"])

print("Confusion Matrix:")
print(knn_test_scores["confusion_matrix"])

print("\nPer-domain:")
knn_test_scores["per_domain"]

k-NN - FINAL TEST
Macro-F1: 0.8648157404269738
Accuracy: 0.8751073487915593


,precision,recall,f1,support
class,,,,
human,0.964433,0.847363,0.902115,5536
ai,0.742927,0.933843,0.827516,2615


Confusion Matrix:
[[4691  845]
 [ 173 2442]]

Per-domain:


{'finance': {'macro_f1': 0.7748998664886515,
  'accuracy': 0.7912218268090154,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.990991  0.558376  0.714286      394
  ai      0.719807  0.995546  0.835514      449,
  'confusion_matrix': array([[220, 174],
         [  2, 447]])},
 'medicine': {'macro_f1': 0.9570799872964022,
  'accuracy': 0.9573643410852714,
  'per_class':        precision   recall        f1  support
  class                                       
  human   1.000000  0.91129  0.953586      124
  ai      0.924138  1.00000  0.960573      134,
  'confusion_matrix': array([[113,  11],
         [  0, 134]])},
 'open_qa': {'macro_f1': 0.8012529597474349,
  'accuracy': 0.8565400843881856,
  'per_class':        precision    recall        f1  support
  class                                        
  human   0.742857  0.655462  0.696429      119
  ai      0.888889  0.923944  0.906077      355,
  'confusion_matri

### k-NN Test Results

The k-NN classifier achieved:

- **Macro-F1: 0.8648**
- **Accuracy: 0.8751**
- **Human Precision: 0.9644**
- **Human Recall: 0.8474**
- **AI Precision: 0.7429**
- **AI Recall: 0.9338**

k-NN detects AI-generated text more aggressively than the Decision Tree, achieving higher AI recall. However, this also results in substantially more human-written examples being incorrectly classified as AI.

## 7. Final Model Comparison

| Model | Macro-F1 | Accuracy | Human Precision | Human Recall | AI Precision | AI Recall |
|---|---:|---:|---:|---:|---:|---:|
| Decision Tree | 0.8918 | 0.9049 | 0.9362 | 0.9229 | 0.8415 | 0.8669 |
| k-NN | 0.8648 | 0.8751 | 0.9644 | 0.8474 | 0.7429 | 0.9338 |

### Conclusion

The **Decision Tree performed better overall**, achieving the higher macro-F1 and accuracy while maintaining a stronger balance between human and AI classification.

The **k-NN model achieved higher AI recall**, identifying approximately 93% of AI-generated examples. However, it also incorrectly classified substantially more human-written examples as AI.

The Decision Tree therefore provides the stronger overall classifier of the two models tested here.

Both models showed meaningful performance differences across HC3 domains. Medicine was consistently one of the easiest domains to classify, while wiki_csai was one of the most difficult. These differences suggest that AI-text detection performance depends partly on the style and subject matter of the input text.